# 04. Lecke - Eszközhasználati tervezési minta

Ebben a leckében megismerheted a Microsoft Agent Framework (Python) által használt AI ügynökök **Eszközhasználati** tervezési mintáját. Lefedjük:

- Függvényeszközök definiálása az `@tool` dekorátorral és típusos paraméterekkel
- Eszköz sémák biztosítása, hogy a modell tudja, mit csinál minden eszköz
- Az eszközvégrehajtás vezérlése az `approval_mode` segítségével
- **Strukturált output** visszaadása Pydantic modellekkel és `response_format`-tal

A szcenárió egy **utazási foglalási ügynök**, amely képes célpontokat keresni, elérhetőséget ellenőrizni és járatinformációkat lekérni.


## Beállítás


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Eszközök definiálása az @tool dekorátorral

Az `@tool` dekorátor egy egyszerű Python függvényt eszközzé alakít, amelyet egy ügynök meghívhat.
Fontos pontok:

- A **docstring** válik az eszköz leírásává, amelyet a modell lát.
- A **típus annotációk** (beleértve az `Annotated` leírásokat is) határozzák meg az eszköz sémáját.
- Az `approval_mode` szabályozza, hogy a felhasználónak jóvá kell-e hagynia minden hívást a végrehajtás előtt.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Több Eszközzel rendelkező Ügynök létrehozása

Add át az összes három eszközt az ügyfélnek, hogy a modell bármelyiket használhassa a felhasználó kérdésének megválaszolásához.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Strukturált kimenet eszközökkel

A `response_format` Pydantic modellre állításával az ügynököt arra kényszerítjük, hogy jól típusozott JSON objektumot adjon vissza szabad szöveg helyett. Ez hasznos, ha a későbbi kódnak programozottan kell feldolgoznia az eredményt.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Eszköz jóváhagyási minták

A `@tool` paraméterében az `approval_mode` szabályozza, hogy az eszközhívások emberi jóváhagyást igényelnek-e a végrehajtás előtt:

| Mód | Viselkedés |
|---|---|
| `"never_require"` | Az eszköz automatikusan fut — nem szükséges felhasználói megerősítés. |
| `"always_require"` | Minden hívást a felhasználónak jóvá kell hagynia a végrehajtás előtt. |

Használd az `"always_require"` módot olyan eszközöknél, amelyek mellékhatásokkal járnak (például repülőjegy foglalása, bankkártya terhelése), hogy az emberi felügyelet megmaradjon.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Összefoglaló

Ebben a leckében megtanultad, hogyan:

1. **Eszközöket definiálj** az `@tool` dekorátor segítségével, típusos paraméterekkel és docstringekkel, amelyek az eszköz sémájaként szolgálnak.
2. **Több eszközt kombinálj**, hogy az ügynök sorban hívhassa azokat összetett kérdések megválaszolására.
3. **Strukturált kimenetet adj vissza** úgy, hogy Pydantic modellt adsz meg `response_format`-ként.
4. **Irányítsd az eszköz engedélyezését** az `approval_mode` segítségével, hogy érzékeny műveleteknél emberi beavatkozás legyen.

Ezek a minták megalapozzák a megbízható, éles használatra kész ügynökök felépítését, amelyek biztonságosan képesek külső rendszerekkel kommunikálni.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
